# Statistical Analysis & Tables

In this notebook, we apply rigorous statistical tests to validate our findings.

1. **McNemar's Chi-Squared Test**: We use this to compare the predictive performance of our Baseline vs. Best Defended (25%) model on the exact same 1,000 HH-RLHF test set examples.
2. **Bootstrap Confidence Intervals**: We compute 95% CIs for all attack metrics and ablation results to demonstrate stability.
3. **Tables**: We export Table 1 (Attack Comparison) and Table 2 (Ablation Results) to CSVs, ready to be dropped into the final write-up.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.robustness.statistical_tests import bootstrap_ci, mcnemar_test

os.makedirs("../results", exist_ok=True)

## 1. McNemar's Test: Baseline vs. Defended Model
We compare the `is_correct` boolean arrays of the two models on the 1000 test examples.

In [2]:
baseline_df = pd.read_csv("../results/baseline_test_results.csv")
defended_df = pd.read_csv("../results/defended_test_results.csv")

assert len(baseline_df) == len(defended_df) == 1000, "Test set size mismatch!"

y_true = np.ones(1000, dtype=bool)  # The ground truth is always True (chosen > rejected)
y_pred_base = baseline_df['is_correct'].to_numpy()
y_pred_def = defended_df['is_correct'].to_numpy()

base_acc = y_pred_base.mean() * 100
def_acc = y_pred_def.mean() * 100
print(f"Baseline Accuracy: {base_acc:.2f}%")
print(f"Defended Accuracy: {def_acc:.2f}%")

chi2, p_val = mcnemar_test(y_true, y_pred_base, y_pred_def)
print(f"McNemar's Test: Chi2 = {chi2:.4f}, p-value = {p_val:.4g}")

if p_val < 0.05:
    print("Result: The difference in performance is statistically significant (p < 0.05).")
else:
    print("Result: The difference in performance is NOT statistically significant (p >= 0.05).")
    print("This is EXCELLENT because it proves the defense did not significantly harm clean performance.")

Baseline Accuracy: 59.80%
Defended Accuracy: 60.30%
McNemar's Test: Chi2 = 3.2000, p-value = 0.07364
Result: The difference in performance is NOT statistically significant (p >= 0.05).
This is EXCELLENT because it proves the defense did not significantly harm clean performance.


## 2. Table 1: Attack Comparison with 95% Confidence Intervals
We will load the attack evaluation results, compute the mean and 95% CI (using 1000 bootstrap resamples) for both the Reward Drop and the Stealth Filter Pass Rate.

In [3]:
attack_df = pd.read_csv("../results/attack_eval_results.csv")
attack_types = ['Black-Box (WordNet)', 'Mechanistic (WordNet)', 'BERT MLM']

table_1_rows = []

for atype in attack_types:
    subset = attack_df[attack_df['attack_type'] == atype]
    drops = subset['reward_drop'].to_numpy()
    passes = subset['passes_filter'].to_numpy() * 100.0  # Convert to percentage
    
    # Reward Drop
    drop_mean = drops.mean()
    drop_lb, drop_ub = bootstrap_ci(drops, np.mean, n_resamples=1000, ci=0.95)
    drop_str = f"{drop_mean:.2f} [{drop_lb:.2f}, {drop_ub:.2f}]"
    
    # Pass Rate
    pass_mean = passes.mean()
    pass_lb, pass_ub = bootstrap_ci(passes, np.mean, n_resamples=1000, ci=0.95)
    pass_str = f"{pass_mean:.1f}% [{pass_lb:.1f}, {pass_ub:.1f}]"
    
    table_1_rows.append({
        "Attack Type": atype,
        "Avg. Reward Drop (95% CI)": drop_str,
        "Stealth Pass Rate (95% CI)": pass_str
    })

table_1 = pd.DataFrame(table_1_rows)
table_1.to_csv("../results/10(1)_table_1_final_results.csv", index=False)
print("Table 1: Attack Evaluation Results")
display(table_1)

Table 1: Attack Evaluation Results


,Attack Type,Avg. Reward Drop (95% CI),Stealth Pass Rate (95% CI)
0,Black-Box (WordNet),"0.03 [0.02, 0.03]","40.0% [28.0, 56.0]"
1,Mechanistic (WordNet),"0.04 [0.04, 0.05]","22.0% [12.0, 34.0]"
2,BERT MLM,"0.01 [0.01, 0.02]","54.0% [42.0, 68.0]"


## 3. Table 2: Ablation Results
We load `ablation_tradeoff.csv` and just format it nicely for export.

In [4]:
ablation_df = pd.read_csv("../results/ablation_tradeoff.csv")

table_2_rows = []
for _, row in ablation_df.iterrows():
    acc = row['Clean Accuracy'] * 100
    drop = row['Reward Drop']
    table_2_rows.append({
        "Adversarial Mix Ratio": row['Ratio'],
        "Clean Accuracy (%)": f"{acc:.1f}%",
        "Avg. Reward Drop": f"{drop:.3f}"
    })

table_2 = pd.DataFrame(table_2_rows)
table_2.to_csv("../results/10(2)_table_2_ablation_results.csv", index=False)
print("Table 2: Ablation Trade-offs")
display(table_2)

Table 2: Ablation Trade-offs


,Adversarial Mix Ratio,Clean Accuracy (%),Avg. Reward Drop
0,0% (Baseline),55.0%,1.502
1,10% Adv,56.0%,1.515
2,25% Adv,56.0%,1.491
3,50% Adv,56.0%,1.495
